# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM

**Βήματα:**
1. Εκπαίδευση μοντέλων
2. Hyperparameter tuning
3. Cross-validation
4. Προβλέψεις

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier, LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("Αρχικοποίηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Modeling_CV") \
    .master("local[*]") \
    .getOrCreate()

print("Φόρτωση δεδομένων Gold Layer...")
train_gold = spark.read.parquet("../data/train_gold.parquet")
test_gold = spark.read.parquet("../data/test_gold.parquet")

# Ορίζουμε τον κοινό Evaluator για το Cross Validation (Βελτιστοποίηση βάσει ROC AUC)
roc_evaluator = BinaryClassificationEvaluator(labelCol="stroke", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# ==========================================
# ΜΟΝΤΕΛΟ 1: Random Forest με Cross-Validation
# ==========================================
print("\nΕκπαίδευση & Tuning Random Forest...")
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", seed=42)

# Grid Υπερπαραμέτρων
rf_paramGrid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10])
    .build())

# Ορισμός 3-fold Cross Validator
rf_cv = CrossValidator(estimator=rf,
                       estimatorParamMaps=rf_paramGrid,
                       evaluator=roc_evaluator,
                       numFolds=3)

rf_cv_model = rf_cv.fit(train_gold)
rf_best_model = rf_cv_model.bestModel # Κρατάμε το καλύτερο μοντέλο
rf_preds = rf_best_model.transform(test_gold)

# ==========================================
# ΜΟΝΤΕΛΟ 2: LinearSVC με Cross-Validation
# ==========================================
print("Εκπαίδευση & Tuning Support Vector Machine (SVM)...")
svm = LinearSVC(featuresCol="features", labelCol="stroke")

# Grid Υπερπαραμέτρων
svm_paramGrid = (ParamGridBuilder()
    .addGrid(svm.maxIter, [50, 100])
    .addGrid(svm.regParam, [0.01, 0.1])
    .build())

svm_cv = CrossValidator(estimator=svm,
                        estimatorParamMaps=svm_paramGrid,
                        evaluator=roc_evaluator,
                        numFolds=3)

svm_cv_model = svm_cv.fit(train_gold)
svm_best_model = svm_cv_model.bestModel
svm_preds = svm_best_model.transform(test_gold)

# ==========================================
# ΑΠΟΘΗΚΕΥΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ & FEATURE IMPORTANCE
# ==========================================
importances = rf_best_model.featureImportances
print(f"\nΒέλτιστο RF Max Depth: {rf_best_model._java_obj.getMaxDepth()}")
print(f"Βέλτιστο RF Num Trees: {rf_best_model._java_obj.getNumTrees()}")
print(f"Feature Importances (Top): {importances}")

print("\nΑποθήκευση των αποτελεσμάτων σε parquet...")
rf_preds.write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.write.mode("overwrite").parquet("../data/svm_predictions.parquet")